## MT5

In [ ]:
# ============================================================
# 1. IMPORTACIONES
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)

import sacrebleu


# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

MODEL_NAME = "google/mt5-base"

OUTPUT_BASE = "./results_mt5_conventional"

MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 2e-5

# Semillas que utilizaremos
SEEDS = [42, 123, 2026]

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


# ============================================================
# 3. CARGAR DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files={
        "train": "train_kichwa_es.jsonl",
        "validation": "valid_kichwa_es.jsonl",
        "test": "test_kichwa_es.jsonl"
    }
)

print(dataset)

print("\nEjemplo original:")
print(dataset["train"][0])


# ============================================================
# 4. PREPARAR DATASET PARA MT5 CONVENCIONAL
# ============================================================


def format_data(example):

    return {
        "input_text": example["input"],
        "target_text": example["output"]
    }


dataset = dataset.map(format_data)

print("\nEjemplo preparado:")
print(dataset["train"][0])


# ============================================================
# 5. TOKENIZER Y MODELO
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("\nModelo cargado:", MODEL_NAME)


# ============================================================
# 6. PREPROCESAMIENTO
# ============================================================

def preprocess(example):

    model_inputs = tokenizer(
        example["input_text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example["target_text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]

    labels_ids = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels_ids
    ]

    model_inputs["labels"] = labels_ids

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess,
    batched=False
)

print("\nTokenización completada.")


# ============================================================
# 7. FUNCIÓN PARA FIJAR LA SEMILLA
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"\nSemilla configurada: {seed}")


# ============================================================
# 8. FUNCIÓN DE TRADUCCIÓN
# ============================================================

def traducir(
    texto,
    model,
    tokenizer,
    max_len=80
):

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            max_length=max_len,

            num_beams=5,

            early_stopping=True,

            no_repeat_ngram_size=2
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


# ============================================================
# 9. FUNCIÓN DE EVALUACIÓN
# ============================================================

def evaluar_modelo(
    model,
    tokenizer,
    dataset_test,
    seed,
    output_dir
):

    print("\n" + "=" * 60)
    print(f"EVALUACIÓN MT5 CONVENCIONAL - SEED {seed}")
    print("=" * 60)

    entradas = []
    referencias = []
    predicciones = []


    # --------------------------------------------------------
    # Generar predicciones
    # --------------------------------------------------------

    for example in dataset_test:

        entrada = example["input"]
        referencia = example["output"]

        prediccion = traducir(
            entrada,
            model,
            tokenizer
        )

        entradas.append(entrada)
        referencias.append(referencia)
        predicciones.append(prediccion)


    print("\nPredicciones generadas correctamente.")


    # --------------------------------------------------------
    # SACREBLEU BLEU
    # --------------------------------------------------------

    bleu_metric = sacrebleu.metrics.BLEU(
        smooth_method="exp",
        smooth_value=None,
        effective_order=False,
        max_ngram_order=4
    )

    bleu = bleu_metric.corpus_score(
        predicciones,
        [referencias]
    )


    # --------------------------------------------------------
    # CHRF++
    # --------------------------------------------------------

    chrf_metric = sacrebleu.metrics.CHRF(
        word_order=2
    )

    chrf = chrf_metric.corpus_score(
        predicciones,
        [referencias]
    )


    # --------------------------------------------------------
    # RESULTADOS
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("RESULTADOS")
    print("=" * 60)

    print(f"Modelo: {MODEL_NAME}")
    print(f"Seed: {seed}")

    print(f"\nBLEU: {bleu.score}")
    print(f"chrF++: {chrf.score}")


    # --------------------------------------------------------
    # SACREBLEU SIGNATURE
    # --------------------------------------------------------

    print("\nSacreBLEU BLEU signature:")
    print(bleu_metric.get_signature())

    print("\nSacreBLEU chrF++ signature:")
    print(chrf_metric.get_signature())


    # --------------------------------------------------------
    # GUARDAR PREDICCIONES
    # --------------------------------------------------------

    df_resultados = pd.DataFrame({

        "seed": [seed] * len(entradas),

        "input_kichwa": entradas,

        "referencia_es": referencias,

        "prediccion_es": predicciones

    })


    predictions_path = os.path.join(
        output_dir,
        f"predicciones_seed_{seed}.csv"
    )

    df_resultados.to_csv(
        predictions_path,
        index=False,
        encoding="utf-8-sig"
    )


    # --------------------------------------------------------
    # GUARDAR RESULTADOS
    # --------------------------------------------------------

    resultados = {

        "model": MODEL_NAME,

        "model_type": "conventional_fine_tuning",

        "seed": seed,

        "BLEU": bleu.score,

        "chrF++": chrf.score,

        "BLEU_signature": bleu_metric.get_signature(),

        "chrF++_signature": chrf_metric.get_signature()

    }


    results_path = os.path.join(
        output_dir,
        f"resultados_seed_{seed}.csv"
    )


    pd.DataFrame([resultados]).to_csv(
        results_path,
        index=False,
        encoding="utf-8-sig"
    )


    return resultados


# ============================================================
# 10. ENTRENAMIENTO + EVALUACIÓN
# ============================================================

todos_los_resultados = []


for SEED in SEEDS:

    print("\n")
    print("#" * 70)
    print(f"# INICIANDO EXPERIMENTO - SEED {SEED}")
    print("#" * 70)


    # --------------------------------------------------------
    # Fijar semilla
    # --------------------------------------------------------

    set_seed(SEED)


    # --------------------------------------------------------
    # Crear modelo NUEVO
    # --------------------------------------------------------

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME
    )


    # --------------------------------------------------------
    # Directorio específico
    # --------------------------------------------------------

    output_dir = os.path.join(
        OUTPUT_BASE,
        f"seed_{SEED}"
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )


    # --------------------------------------------------------
    # TrainingArguments
    # --------------------------------------------------------

    training_args = TrainingArguments(

        output_dir=output_dir,

        per_device_train_batch_size=BATCH_SIZE,

        per_device_eval_batch_size=BATCH_SIZE,

        num_train_epochs=EPOCHS,

        learning_rate=LEARNING_RATE,

        logging_steps=10,

        save_strategy="epoch",

        eval_strategy="epoch",

        report_to="none",

        seed=SEED,

        data_seed=SEED,

        save_total_limit=1,

        load_best_model_at_end=False
    )


    # --------------------------------------------------------
    # Trainer
    # --------------------------------------------------------

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=tokenized_dataset["train"],

        eval_dataset=tokenized_dataset["validation"]
    )


    # --------------------------------------------------------
    # ENTRENAMIENTO
    # --------------------------------------------------------

    print(f"\nEntrenando mT5 con seed {SEED}...")

    trainer.train()


    # --------------------------------------------------------
    # GUARDAR MODELO
    # --------------------------------------------------------

    model_path = os.path.join(
        output_dir,
        "model"
    )

    trainer.save_model(model_path)

    tokenizer.save_pretrained(
        model_path
    )

    print(
        f"\nModelo guardado en: {model_path}"
    )


    # --------------------------------------------------------
    # MOVER A CPU/GPU
    # --------------------------------------------------------

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model.to(device)

    print(
        "Modelo en:",
        model.device
    )


    # --------------------------------------------------------
    # EVALUAR
    # --------------------------------------------------------

    resultado = evaluar_modelo(

        model=model,

        tokenizer=tokenizer,

        dataset_test=dataset["test"],

        seed=SEED,

        output_dir=output_dir
    )


    todos_los_resultados.append(
        resultado
    )


# ============================================================
# 11. RESULTADOS DE LAS TRES SEMILLAS
# ============================================================

df_resultados_finales = pd.DataFrame(
    todos_los_resultados
)


print("\n")
print("=" * 70)
print("RESULTADOS FINALES - MT5 CONVENCIONAL")
print("=" * 70)

print(
    df_resultados_finales[
        [
            "seed",
            "BLEU",
            "chrF++"
        ]
    ]
)


# ============================================================
# 12. MEDIA Y DESVIACIÓN ESTÁNDAR
# ============================================================

print("\n")
print("=" * 70)
print("VARIABILIDAD ENTRE SEMILLAS")
print("=" * 70)


bleu_mean = df_resultados_finales["BLEU"].mean()
bleu_std = df_resultados_finales["BLEU"].std()

chrf_mean = df_resultados_finales["chrF++"].mean()
chrf_std = df_resultados_finales["chrF++"].std()


print(
    f"BLEU: {bleu_mean:.4f} ± {bleu_std:.4f}"
)

print(
    f"chrF++: {chrf_mean:.4f} ± {chrf_std:.4f}"
)


# ============================================================
# 13. GUARDAR RESULTADOS CONSOLIDADOS
# ============================================================

df_resultados_finales.to_csv(
    os.path.join(
        OUTPUT_BASE,
        "resultados_mt5_todas_las_semillas.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nResultados consolidados guardados correctamente."
)

PyTorch: 2.12.0+cpu
CUDA disponible: False
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2300
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 287
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 288
    })
})

Ejemplo original:
{'instruction': 'Traduce del Kichwa al Español', 'input': 'Ñawpa pachapa shamuk pachapa hampikkuna', 'output': 'Píldoras para antes y para después.'}

Ejemplo preparado:
{'instruction': 'Traduce del Kichwa al Español', 'input': 'Ñawpa pachapa shamuk pachapa hampikkuna', 'output': 'Píldoras para antes y para después.', 'input_text': 'Ñawpa pachapa shamuk pachapa hampikkuna', 'target_text': 'Píldoras para antes y para después.'}


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 27910.92it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Modelo cargado: google/mt5-base

Tokenización completada.


######################################################################
# INICIANDO EXPERIMENTO - SEED 42
######################################################################

Semilla configurada: 42


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 97041.33it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Entrenando mT5 con seed 42...


...


Entrenando mT5 con seed 2026...


c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,5.371033,4.213869
2,4.130062,3.350827
3,3.753834,3.154745
4,3.829282,3.122793
5,3.964534,3.115574


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.58s/it]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.79s/it]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.50s/it]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█


Modelo guardado en: ./results_mt5_conventional\seed_2026\model
Modelo en: cpu

EVALUACIÓN MT5 CONVENCIONAL - SEED 2026

Predicciones generadas correctamente.

RESULTADOS
Modelo: google/mt5-base
Seed: 2026

BLEU: 0.2691400098375429
chrF++: 5.737123634801463

SacreBLEU BLEU signature:
nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0

SacreBLEU chrF++ signature:
nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0


RESULTADOS FINALES - MT5 CONVENCIONAL
   seed      BLEU     chrF++
0    42  0.684546  11.143722
1   123  0.534961   8.886505
2  2026  0.269140   5.737124


VARIABILIDAD ENTRE SEMILLAS
BLEU: 0.4962 ± 0.2104
chrF++: 8.5891 ± 2.7155

Resultados consolidados guardados correctamente.


## FLAN-T5 BASE - INSTRUCTION FINE-TUNING

In [7]:
# ============================================================
# FLAN-T5 BASE - INSTRUCTION FINE-TUNING
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)
import sacrebleu

# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

MODEL_NAME = "google/flan-t5-base"
OUTPUT_BASE = "./results_flan_t5_finetuned"

MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 5e-5

SEEDS = [42, 123, 2026]

print("PyTorch Version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


# ============================================================
# 2. CARGAR DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files={
        "train": "train_kichwa_es.jsonl",
        "validation": "valid_kichwa_es.jsonl",
        "test": "test_kichwa_es.jsonl"
    }
)

print(dataset)


# ============================================================
# 3. TOKENIZER Y PREPROCESAMIENTO DE INSTRUCCIONES
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_flan(example):
    # En FLAN-T5 las entradas ya contienen la instrucción si vienen de jsonl formateado,
    # o tomamos directamente los campos "input" y "output"
    model_inputs = tokenizer(
        example["input"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example["output"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]
    labels_ids = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_dataset = dataset.map(preprocess_flan, batched=False)
print("\nDataset tokenizado para FLAN-T5 correctamente.")


# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"\nSemilla fijada: {seed}")


def traducir(texto, model, tokenizer, max_len=80):
    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_len,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def evaluar_modelo(model, tokenizer, dataset_test, seed, output_dir):
    print("\n" + "=" * 60)
    print(f"EVALUANDO FLAN-T5 FINE-TUNED - SEED {seed}")
    print("=" * 60)

    entradas, referencias, predicciones = [], [], []

    for example in dataset_test:
        entrada = example["input"]
        referencia = example["output"]
        prediccion = traducir(entrada, model, tokenizer)

        entradas.append(entrada)
        referencias.append(referencia)
        predicciones.append(prediccion)

    # Cálculo de métricas SacreBLEU
    bleu_metric = sacrebleu.metrics.BLEU(
        smooth_method="exp",
        smooth_value=None,
        effective_order=False,
        max_ngram_order=4
    )
    bleu = bleu_metric.corpus_score(predicciones, [referencias])

    chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
    chrf = chrf_metric.corpus_score(predicciones, [referencias])

    print(f"BLEU: {bleu.score:.4f}")
    print(f"chrF++: {chrf.score:.4f}")

    # Guardar predicciones en CSV por semilla
    df_predicciones = pd.DataFrame({
        "seed": [seed] * len(entradas),
        "input_kichwa": entradas,
        "referencia_es": referencias,
        "prediccion_es": predicciones
    })
    
    pred_path = f"predicciones_flan_t5_finetuned_seed{seed}.csv"
    df_predicciones.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print(f"Predicciones guardadas en: {pred_path}")

    return {
        "model": "FLAN-T5-Base",
        "model_type": "instruction_fine_tuning",
        "seed": seed,
        "BLEU": bleu.score,
        "chrF++": chrf.score,
        "BLEU_signature": bleu_metric.get_signature(),
        "chrF++_signature": chrf_metric.get_signature()
    }


# ============================================================
# 5. BUCLE DE ENTRENAMIENTO Y EVALUACIÓN POR SEMILLA
# ============================================================

todos_los_resultados = []

for SEED in SEEDS:
    print("\n" + "#" * 70)
    print(f"# INICIANDO ENTRENAMIENTO FLAN-T5 - SEED {SEED}")
    print("#" * 70)

    set_seed(SEED)

    # Descargar modelo base limpio en cada iteración
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    output_dir = os.path.join(OUTPUT_BASE, f"seed_{SEED}")
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        save_total_limit=1,
        load_best_model_at_end=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"]
    )

    print(f"\nEntrenando FLAN-T5 para semilla {SEED}...")
    trainer.train()

    # Guardar modelo entrenado de esta semilla
    model_save_path = os.path.join(output_dir, "model")
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    # Mover a GPU para inferencia
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Evaluar
    resultado = evaluar_modelo(
        model=model,
        tokenizer=tokenizer,
        dataset_test=dataset["test"],
        seed=SEED,
        output_dir=output_dir
    )

    todos_los_resultados.append(resultado)

    # Limpiar VRAM de GPU antes de la siguiente semilla
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 6. CONSOLIDADO DE RESULTADOS
# ============================================================

df_final = pd.DataFrame(todos_los_resultados)

print("\n" + "=" * 70)
print("RESULTADOS FINALES FLAN-T5 FINE-TUNED")
print("=" * 70)
print(df_final[["seed", "BLEU", "chrF++"]])

bleu_mean, bleu_std = df_final["BLEU"].mean(), df_final["BLEU"].std()
chrf_mean, chrf_std = df_final["chrF++"].mean(), df_final["chrF++"].std()

print("\n" + "=" * 70)
print("VARIABILIDAD ENTRE SEMILLAS")
print("=" * 70)
print(f"BLEU:   {bleu_mean:.4f} ± {bleu_std:.4f}")
print(f"chrF++: {chrf_mean:.4f} ± {chrf_std:.4f}")

df_final.to_csv(
    os.path.join(OUTPUT_BASE, "resultados_flan_t5_todas_las_semillas.csv"),
    index=False,
    encoding="utf-8-sig"
)

PyTorch Version: 2.12.0+cpu
CUDA disponible: False
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2300
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 287
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 288
    })
})


Map: 100%|██████████| 288/288 [00:00<00:00, 2726.01 examples/s]



Dataset tokenizado para FLAN-T5 correctamente.

######################################################################
# INICIANDO ENTRENAMIENTO FLAN-T5 - SEED 42
######################################################################

Semilla fijada: 42


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 9596.55it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Entrenando FLAN-T5 para semilla 42...


...


Entrenando FLAN-T5 para semilla 2026...


c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.186015,1.989020
2,2.216799,1.903181
3,1.944135,1.868397
4,1.962833,1.853534
5,2.021571,1.849827


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.50it/s]
c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█


EVALUANDO FLAN-T5 FINE-TUNED - SEED 2026
BLEU: 3.6669
chrF++: 21.6935
Predicciones guardadas en: predicciones_flan_t5_finetuned_seed2026.csv

RESULTADOS FINALES FLAN-T5 FINE-TUNED
   seed      BLEU     chrF++
0    42  3.371584  21.874067
1   123  3.344440  21.150510
2  2026  3.666915  21.693516

VARIABILIDAD ENTRE SEMILLAS
BLEU:   3.4610 ± 0.1789
chrF++: 21.5727 ± 0.3766


## FLAN-T5 BASE - WITHOUT INSTRUCTION FINE-TUNING

In [ ]:
# ============================================================
# FLAN-T5 BASE (WITHOUT FINE-TUNING)
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import sacrebleu

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

MODEL_NAME = "google/flan-t5-base"

SEEDS = [42, 123, 2026]

MAX_INPUT_LENGTH = 128
MAX_GENERATION_LENGTH = 80

BEAM_SIZE = 5
NO_REPEAT_NGRAM_SIZE = 2
EARLY_STOPPING = True

OUTPUT_DIR = "./resultados_flant5_base_sin_finetuning"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. CARGAR DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files={
        "train": "train_kichwa_es.jsonl",
        "validation": "valid_kichwa_es.jsonl",
        "test": "test_kichwa_es.jsonl"
    }
)

print(dataset)
print("\nEjemplo del dataset:")
print(dataset["test"][0])


# ============================================================
# 3. CARGAR TOKENIZER Y MODELO BASE
# ============================================================

print("\nCargando modelo:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


# ============================================================
# 4. CONFIGURAR DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("\nModelo en:", model.device)


# ============================================================
# 5. FUNCIÓN DE TRADUCCIÓN
# ============================================================

def traducir(texto):

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=MAX_GENERATION_LENGTH,
            num_beams=BEAM_SIZE,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            early_stopping=EARLY_STOPPING
        )

    traduccion = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return traduccion


# ============================================================
# 6. EVALUACIÓN PARA CADA SEMILLA
# ============================================================

resultados = []

for seed in SEEDS:

    print("\n" + "=" * 60)
    print(f"EVALUANDO FLAN-T5 BASE (SIN FINE-TUNING) - SEED {seed}")
    print("=" * 60)

    # --------------------------------------------------------
    # Fijar Semillas
    # --------------------------------------------------------
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # --------------------------------------------------------
    # Generar Predicciones
    # --------------------------------------------------------
    print("\nGenerando predicciones...")

    predicciones = []
    referencias = []
    entradas = []

    for i, example in enumerate(dataset["test"]):

        entrada = example["input"]
        referencia = example["output"]

        prediccion = traducir(entrada)

        entradas.append(entrada)
        referencias.append(referencia)
        predicciones.append(prediccion)

        if (i + 1) % 25 == 0:
            print(f"Procesadas {i + 1}/{len(dataset['test'])}")

    print("\nPredicciones generadas correctamente.")

    # --------------------------------------------------------
    # Guardar CSV de Predicciones por Semilla
    # --------------------------------------------------------
    output_csv = os.path.join(
        OUTPUT_DIR,
        f"predicciones_flant5_base_sin_finetuning_seed{seed}.csv"
    )

    df_resultados = pd.DataFrame({
        "input_kichwa": entradas,
        "referencia_es": referencias,
        "prediccion_es": predicciones
    })

    df_resultados.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nArchivo guardado: {output_csv}")

    # --------------------------------------------------------
    # BLEU y chrF++ usando Objetos Métrica
    # --------------------------------------------------------
    bleu_metric = sacrebleu.metrics.BLEU()
    bleu = bleu_metric.corpus_score(
        predicciones,
        [referencias]
    )

    chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
    chrf = chrf_metric.corpus_score(
        predicciones,
        [referencias]
    )

    bleu_signature = str(bleu_metric.get_signature())
    chrf_signature = str(chrf_metric.get_signature())

    # --------------------------------------------------------
    # Mostrar Resultados Individuales
    # --------------------------------------------------------
    print("\n==============================")
    print(f"FLAN-T5 BASE (seed {seed})")
    print("==============================")

    print(f"BLEU: {bleu.score}")
    print(f"chrF++: {chrf.score}")

    print("\nSacreBLEU signature (BLEU):")
    print(bleu_signature)

    print("\nSacreBLEU signature (chrF++):")
    print(chrf_signature)

    # Guardar métricas en lista general
    resultados.append({
        "model": "FLAN-T5 Base (sin fine-tuning)",
        "seed": seed,
        "BLEU": bleu.score,
        "chrF++": chrf.score,
        "signature_BLEU": bleu_signature,
        "signature_chrF": chrf_signature
    })


# ============================================================
# 7. TABLA FINAL Y ESTADÍSTICAS (MEDIA Y DESVIACIÓN ESTÁNDAR)
# ============================================================

df_resumen = pd.DataFrame(resultados)

print("\n" + "=" * 60)
print("RESULTADOS FINALES - FLAN-T5 BASE (SIN FINE-TUNING)")
print("=" * 60)

print(
    df_resumen[
        ["model", "seed", "BLEU", "chrF++"]
    ].to_string(index=False)
)

media_bleu = df_resumen["BLEU"].mean()
std_bleu = df_resumen["BLEU"].std()

media_chrf = df_resumen["chrF++"].mean()
std_chrf = df_resumen["chrF++"].std()

print("\n" + "=" * 60)
print("MEDIA Y DESVIACIÓN ESTÁNDAR")
print("=" * 60)

print(f"BLEU   : {media_bleu:.4f} ± {std_bleu:.4f}")
print(f"chrF++ : {media_chrf:.4f} ± {std_chrf:.4f}")


# ============================================================
# 8. GUARDAR RESUMEN DE EXPERIMENTOS
# ============================================================

summary_file = os.path.join(
    OUTPUT_DIR,
    "resumen_flant5_base_sin_finetuning_multiseed.csv"
)

df_resumen.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nResumen general guardado en: {summary_file}")


# ============================================================
# 9. EJEMPLOS DE LA ÚLTIMA SEMILLA
# ============================================================

print("\n" + "=" * 60)
print("EJEMPLOS DE PREDICCIONES (ÚLTIMA SEMILLA)")
print("=" * 60)

for i in range(min(10, len(predicciones))):

    print("\nKichwa:")
    print(entradas[i])

    print("\nReferencia:")
    print(referencias[i])

    print("\nPredicción FLAN-T5 Base:")
    print(predicciones[i])

    print("-" * 70)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2300
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 287
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 288
    })
})

Ejemplo del dataset:
{'instruction': 'Traduce del Kichwa al Español', 'input': 'Imanallatak kanki, shamupay, yaykuripay, tiyaripay.', 'output': '¿Cómo estás?, venga, entre, siéntese.'}

Cargando modelo: google/flan-t5-base
